# Plot IR-luminosity MIRI

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prospector_utils.analysis import get_dust_luminosity
from astropy.io import fits


table_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'



with fits.open(table_path) as hdul:
        galaxy_ids = hdul[1].data['ID']

all_ids = set(galaxy_ids)

no_spec = set([9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990])
intermediate_ap = set([7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452])
bl_agn = set([12020, 18977])
high_z = set([1909, 11247, 7696, 21026])

exclude = no_spec | bl_agn | intermediate_ap | high_z

diff = all_ids.difference(exclude)


sourcephotonly = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/sourcephotonly/flux_scaled'
appphot = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/appphot_only_wMIRI/flux_scaled'
out_dir = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/miri_comparison/'

# 1. Extraction Loop -> Collect records into a list of dicts
records = []

for objid in diff:
    try:
        res_nomiri = get_dust_luminosity(objid, sourcephotonly)
        res_miri = get_dust_luminosity(objid, appphot)

        records.append({
            'objid': objid,
            # MIRI values (X)
            'log_L_ir_miri': res_miri['log_L_ir_median'],
            'err_low_miri': res_miri['err_low_dex'],
            'err_high_miri': res_miri['err_high_dex'],
            'logmass_miri': res_miri['logmass'],
            'dust_miri': res_miri['dust'],
            'sfr_miri': res_miri['sfr'],
            # No-MIRI values (Y)
            'log_L_ir_nomiri': res_nomiri['log_L_ir_median'],
            'err_low_nomiri': res_nomiri['err_low_dex'],
            'err_high_nomiri': res_nomiri['err_high_dex'],
            'logmass_nomiri': res_nomiri['logmass'],
            'dust_nomiri': res_nomiri['dust'],
            'sfr_nomiri': res_nomiri['sfr'],
        })

    except Exception as e:
        print(f"⚠️ Skipping galaxy {objid} due to error: {e}")
        continue

# Create DataFrame
df_lum = pd.DataFrame(records)
df_lum.to_csv(os.path.join(out_dir, 'LIR_results.csv'), index=False)

dust_mask = df_lum['dust_nomiri'] > 1
df_lum = df_lum[dust_mask]

# 2. Extract arrays directly from DataFrame for plotting
x_log_L = df_lum['log_L_ir_miri'].values
y_log_L = df_lum['log_L_ir_nomiri'].values

# Format asymmetric errors for matplotlib: shape (2, N)
x_err = [df_lum['err_low_miri'].values, df_lum['err_high_miri'].values]
y_err = [df_lum['err_low_nomiri'].values, df_lum['err_high_nomiri'].values]

# 3. Setup Plot
fig, ax = plt.subplots(figsize=(5, 5))

ax.errorbar(
    x_log_L,
    y_log_L,
    xerr=x_err,
    yerr=y_err,
    fmt='o',
    color='#1f77b4',
    alpha=0.7,
    capsize=2,
    markersize=6,
    markeredgecolor='black',
    markeredgewidth=0.5,
    label=r'Blue Jay ($A_V>1$)',
)

# 1:1 Reference Line
all_vals = np.concatenate([x_log_L, y_log_L])
lim_min = np.floor(np.nanmin(all_vals) * 2) / 2 - 0.2
lim_max = np.ceil(np.nanmax(all_vals) * 2) / 2 + 0.2

ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1.2, label='1:1 Line')

# Formatting for A&A Style
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_aspect('equal')

ax.set_xlabel(
    r'$\log_{10}(L_{\mathrm{IR}} / L_\odot)$ [with MIRI]', fontsize=14
)
ax.set_ylabel(
    r'$\log_{10}(L_{\mathrm{IR}} / L_\odot)$ [no MIRI]', fontsize=14
)
ax.set_title(
    r'MIRI Constraints on Integrated $L_{\mathrm{IR}}$', fontsize=14
)

ax.tick_params(labelsize=12, direction='in', top=True, right=True)
ax.legend(fontsize=12, loc='upper left', frameon=True)

plt.tight_layout()

# Save plot & DataFrame if needed
out_plot = os.path.join(out_dir, 'LIR_dusty_nomiri.png')
plt.savefig(out_plot, dpi=300, bbox_inches='tight')

print(f'✅ Comparison plot ready!')
plt.show()

lum_miri = 10**x_log_L
lum_nomiri = 10**y_log_L
ratio = lum_miri/lum_nomiri

print(np.median(ratio))
print(np.mean(ratio))


# Plot ratio over Av

In [ ]:
from scipy.stats import binned_statistic
import matplotlib.ticker as ticker

out_dir = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/miri_comparison/'

df_lum = pd.read_csv(os.path.join(out_dir, 'LIR_results.csv'))

# 2. Extract arrays directly from DataFrame for plotting
x_log_L = df_lum['log_L_ir_miri'].values
y_log_L = df_lum['log_L_ir_nomiri'].values

ratio = 10**(x_log_L - y_log_L) # miri over no_miri

dust = df_lum['dust_nomiri'].values

# 3. Setup Plot
fig, ax = plt.subplots(figsize=(4.5, 4.5))

# 1 ratio Reference Line
ax.axhline(
    y=1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.8, zorder=1
)


ax.set_xlabel(
    r'$A_V$', fontsize=16
)
ax.set_ylabel(
    r'$L_{IR}^{MIRI} \,/\, L_{IR}^{no\,MIRI}$', fontsize=16
)

#ax.set_title(
#    r'Luminosity ratios vs $A_V$', fontsize=16
#)

nbins = 5
bin_edges = np.linspace(np.min(dust), np.max(dust), nbins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

medians, _, _ = binned_statistic(
    dust, ratio, statistic='median', bins=bin_edges
)
p16, _, _ = binned_statistic(
    dust, ratio, statistic=lambda x: np.percentile(x, 16), bins=bin_edges
)
p84, _, _ = binned_statistic(
    dust, ratio, statistic=lambda x: np.percentile(x, 84), bins=bin_edges
)

# Plot binned median trend
ax.plot(
    bin_centers,
    medians,
    color='crimson',
    lw=2.0,
    label='Running Median',
    zorder=3,
)
ax.fill_between(
    bin_centers,
    p16,
    p84,
    color='crimson',
    alpha=0.15,
    zorder=2,
    label=r'16--84th percentile',
)

ax.scatter(
    dust,
    ratio,
    color='#1f77b4',
    alpha=0.7,
    label=r'Blue Jay',
)

# 6. Log-Scale & Axis Bounds
ax.set_yscale('log')
ax.set_ylim(0.08, 4.0)  # Handles symmetric factor of ~10 up and down
ax.set_xlim(-0.1, 3.2)


#ax.set_ylim(0, 5)

plt.tight_layout()

# Save plot & DataFrame if needed
out_dir = '/Users/benjamincollins/University/Master/Red_Cardinal/prospector/miri_comparison/'
out_plot = os.path.join(out_dir, 'LIR_log_ratios.png')
plt.savefig(out_plot, dpi=300, bbox_inches='tight')
plt.show()

# Claude Suggestion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import binned_statistic
import numpy as np

fig, ax = plt.subplots(figsize=(4, 3.6))  # single-column A&A width

ax.axhline(y=1.0, color='grey', linestyle='--', linewidth=1.5, zorder=1, label='Unity')

ax.set_xlabel(r'$A_V$', fontsize=12)
ax.set_ylabel(r'$L_{IR}^{MIRI} \,/\, L_{IR}^{no\,MIRI}$', fontsize=12)
ax.tick_params(labelsize=12, direction='in', top=True, right=True) 

nbins = 5
bin_edges = np.linspace(np.min(dust), np.max(dust), nbins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

medians, _, _ = binned_statistic(dust, ratio, statistic='median', bins=bin_edges)
p16, _, _ = binned_statistic(dust, ratio, statistic=lambda x: np.percentile(x, 16), bins=bin_edges)
p84, _, _ = binned_statistic(dust, ratio, statistic=lambda x: np.percentile(x, 84), bins=bin_edges)

ax.plot(bin_centers, medians, color='crimson', lw=1.5, label='Running median', zorder=3, alpha=0.7)
ax.fill_between(bin_centers, p16, p84, color='crimson', alpha=0.15,
                 label=r'$1\sigma$ uncertainty', zorder=2)

ax.scatter(dust, ratio, s=30, facecolor='#1f77b4', edgecolor='black',
           linewidth=0.3, alpha=0.7, zorder=4)

ax.set_ylim(0.08, 15.0)
ax.set_yscale('log')
ax.set_xlim(-0.1, 3.2)

ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()

out_plot_png = os.path.join(out_dir, 'LIR_log_ratios2.png')
plt.savefig(out_plot_png, dpi=300, bbox_inches='tight')
plt.show()